In [0]:
sales_df = spark.read.format("csv")\
    .option("Header","true")\
    .option("inferSchema","true")\
    .load("/Volumes/sql_problems/default/my_volume/day05_sales.csv")

display(sales_df)

In [0]:
sales_df.createOrReplaceTempView("sales")

In [0]:
%sql
WITH lastRevenue AS(
    SELECT *,
        -- Step 1: Get previous year's revenue using LAG()
        LAG(Yearly_Revenue) OVER(
        PARTITION BY Region,product_id
        ORDER BY Year
        ) as lastYear_revenue
    FROM sales
)
-- Step 2: Calculate YoY Growth Percentage
SELECT product_id, year, Yearly_Revenue as revenue,  
    ROUND( 
        (Yearly_Revenue - lastYear_revenue) *100 / lastYear_revenue
        ,2
    ) as yoy_growth_pct
FROM lastRevenue



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, round

# Define the window
windowSpec = Window.partitionBy("Region", "Product_ID").orderBy("Year")

# Apply LAG and calculate growth
result_df = sales_df.withColumn("prev_year_revenue", lag(col("Yearly_Revenue"), 1).over(windowSpec)) \
            .withColumn(
                "yoy_growth_pct",
                round(
                    ((col("Yearly_Revenue") - col("prev_year_revenue")) / col("prev_year_revenue")) * 100
                    , 2)
            )
result_df = result_df.select(
    col("Product_ID"), col("Year"), 
    col("Yearly_Revenue"), col("yoy_growth_pct")
)

display(result_df)